# Tutorial 3: Model Training - Discovering Pressing Patterns

This tutorial demonstrates how to train models to discover pressing patterns. You'll learn:

- How to train GMM zone models for spatial discretization
- How tokenization converts movements to discrete sequences
- How NMF discovers pressing pattern topics
- How to interpret learned topics tactically

## Prerequisites

Complete Tutorials 1-2 to have:
- Extracted build-ups in `data/processed/rm_pressing_tutorial/`
- Features in `features.parquet`

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import NMF
from matplotlib.patches import Ellipse
import pickle

# Model modules
from src.models.gmm_zones import GMMZoneModel, identify_pressers
from src.models.tokenization import tokenize_build_up
from src.models.config import GMMConfig, NMFConfig
from src.features.services.window_loader import WindowLoader
from src.features.services.normalization import normalize_coordinates
from src.features.services.possession import infer_ball_carrier
from src.features.services.utils import prepare_frame_data, time_to_seconds
from src.features.services.metadata import enrich_with_team_id

# Configuration
PROCESSED_ROOT = Path("data/processed/rm_pressing_tutorial")
ZONES_DIR = Path("data/processed/rm_pressing_tutorial_zones")
TOKENS_DIR = Path("data/processed/rm_pressing_tutorial_tokens")
TOPICS_DIR = Path("data/processed/rm_pressing_tutorial_topics")

ZONES_DIR.mkdir(parents=True, exist_ok=True)
TOKENS_DIR.mkdir(parents=True, exist_ok=True)
TOPICS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")

## Part 1: GMM Zone Modeling

### Step 1: Collect Presser Positions

Aggregate initial and target positions of active pressers across all build-ups.

In [ ]:
from tqdm.notebook import tqdm

loader = WindowLoader(PROCESSED_ROOT)
index = loader.index
config = GMMConfig()

all_initial_positions = []
all_target_positions = []

print(f"Collecting presser positions from {len(index)} build-ups...")

for build_up_id in tqdm(index['build_up_id'].tolist()[:50]):  # First 50 for demo
    try:
        # Load and preprocess
        df = loader.load_build_up(build_up_id)
        meta = loader.get_metadata(build_up_id)
        
        df = prepare_frame_data(df)
        df = enrich_with_team_id(df, meta['game_id'])
        df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))
        df_norm = infer_ball_carrier(df_norm, meta.get('opponent_team_id'))
        df_norm['time_seconds'] = df_norm['time'].apply(time_to_seconds)
        
        # Identify active pressers
        pressers = identify_pressers(df_norm, config)
        if len(pressers) == 0:
            continue
        
        # Extract positions at t_init (kick + 1s) and t_target (kick + 5s)
        kick_time = time_to_seconds(str(meta.get('kick_time')))
        t_init = kick_time + 1.0
        t_target = kick_time + 5.0
        
        # Find closest frames
        times = df_norm[['frame', 'time_seconds']].drop_duplicates()
        
        # Initial positions
        init_frame_idx = (times['time_seconds'] - t_init).abs().idxmin()
        init_frame = times.loc[init_frame_idx, 'frame']
        init_df = df_norm[df_norm['frame'] == init_frame]
        
        for pid in pressers:
            p_row = init_df[init_df['player_id'] == pid]
            if not p_row.empty:
                all_initial_positions.append([p_row.iloc[0]['x_norm'], p_row.iloc[0]['y_norm']])
        
        # Target positions
        target_frame_idx = (times['time_seconds'] - t_target).abs().idxmin()
        target_frame = times.loc[target_frame_idx, 'frame']
        target_df = df_norm[df_norm['frame'] == target_frame]
        
        for pid in pressers:
            p_row = target_df[target_df['player_id'] == pid]
            if not p_row.empty:
                all_target_positions.append([p_row.iloc[0]['x_norm'], p_row.iloc[0]['y_norm']])
    
    except Exception as e:
        continue

X_init = np.array(all_initial_positions)
X_target = np.array(all_target_positions)

print(f"\nCollected {len(X_init)} initial positions")
print(f"Collected {len(X_target)} target positions")

### Step 2: Visualize Position Distributions

Plot the raw spatial distributions before GMM fitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Initial positions
ax1.scatter(X_init[:, 0], X_init[:, 1], alpha=0.3, s=10, color='blue')
ax1.set_xlim(-52.5, 52.5)
ax1.set_ylim(-34, 34)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax1.axvline(0, color='gray', linestyle='--', alpha=0.3)
ax1.set_xlabel('X (meters)')
ax1.set_ylabel('Y (meters)')
ax1.set_title(f'Initial Positions (kick + 1s) - {len(X_init)} points')
ax1.grid(alpha=0.3)

# Target positions
ax2.scatter(X_target[:, 0], X_target[:, 1], alpha=0.3, s=10, color='red')
ax2.set_xlim(-52.5, 52.5)
ax2.set_ylim(-34, 34)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax2.axvline(0, color='gray', linestyle='--', alpha=0.3)
ax2.set_xlabel('X (meters)')
ax2.set_ylabel('Y (meters)')
ax2.set_title(f'Target Positions (kick + 5s) - {len(X_target)} points')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation: Target positions are more dispersed (players spread out during pressing)")

### Step 3: Train GMM Zone Models

Fit two GMMs: 8 zones for initial positions, 15 zones for target positions.

In [ ]:
model = GMMZoneModel(config)
model.fit(X_init, X_target)

print("GMM Models Trained:")
print(f"  Initial GMM: {config.n_initial_zones} zones, converged={model.gmm_initial.converged_}")
print(f"  Target GMM: {config.n_target_zones} zones, converged={model.gmm_target.converged_}")

# Save models
model.save(ZONES_DIR)
print(f"\nSaved models to {ZONES_DIR}/")

### Step 4: Visualize Learned Zones

Plot Gaussian ellipses (2σ) overlaid on the pitch to show zone locations and shapes.

In [ ]:
def plot_gmm_zones(gmm, positions, title, ax):
    """Plot GMM zone ellipses on pitch."""
    # Scatter positions
    ax.scatter(positions[:, 0], positions[:, 1], alpha=0.2, s=5, color='gray')
    
    # Plot ellipses for each component
    for i in range(gmm.n_components):
        mean = gmm.means_[i]
        covar = gmm.covariances_[i]
        
        # Compute 2-sigma ellipse
        eigenvalues, eigenvectors = np.linalg.eigh(covar)
        angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
        width, height = 2 * 2 * np.sqrt(eigenvalues)  # 2 standard deviations
        
        ellipse = Ellipse(mean, width, height, angle=angle,
                         alpha=0.4, facecolor=f'C{i}', edgecolor=f'C{i}', linewidth=2)
        ax.add_patch(ellipse)
        
        # Label zone
        ax.text(mean[0], mean[1], f'Z{i}', fontsize=12, ha='center', 
               fontweight='bold', color='white',
               bbox=dict(boxstyle='round', facecolor=f'C{i}', alpha=0.8))
    
    ax.set_xlim(-52.5, 52.5)
    ax.set_ylim(-34, 34)
    ax.axhline(0, color='black', linestyle='-', alpha=0.2)
    ax.axvline(0, color='black', linestyle='-', alpha=0.2)
    ax.set_xlabel('X (meters)')
    ax.set_ylabel('Y (meters)')
    ax.set_title(title)
    ax.grid(alpha=0.3)

# Plot both GMMs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

plot_gmm_zones(model.gmm_initial, X_init, 'Initial Zones (8 zones, kick + 1s)', ax1)
plot_gmm_zones(model.gmm_target, X_target, 'Target Zones (15 zones, kick + 5s)', ax2)

plt.tight_layout()
plt.show()

### Step 5: Interpret Zone Meanings

Examine zone centroids to assign tactical labels.

In [ ]:
print("Initial Zone Centroids (8 zones):")
print("Zone | X (m)  | Y (m)  | Interpretation")
print("-" * 60)
for i in range(config.n_initial_zones):
    x, y = model.gmm_initial.means_[i]
    
    # Interpret position
    depth = "High" if x > 0 else ("Mid" if x > -20 else "Deep")
    lateral = "Left" if y < -10 else ("Center" if y < 10 else "Right")
    
    print(f"  {i}  | {x:6.1f} | {y:6.1f} | {depth} {lateral}")

print("\n" + "="*60)
print("\nTarget Zone Centroids (15 zones):")
print("Zone | X (m)  | Y (m)  | Interpretation")
print("-" * 60)
for i in range(config.n_target_zones):
    x, y = model.gmm_target.means_[i]
    
    depth = "High" if x > 0 else ("Mid" if x > -20 else "Deep")
    lateral = "Left" if y < -10 else ("Center" if y < 10 else "Right")
    
    print(f" {i:2d}  | {x:6.1f} | {y:6.1f} | {depth} {lateral}")

## Part 2: Tokenization

### Step 6: Create Token Matrix

Convert zone transitions to discrete tokens. Each build-up becomes a 120-dimensional token weight vector.

In [ ]:
# Load trained GMMs
with open(ZONES_DIR / "gmm_initial.pkl", "rb") as f:
    gmm_init = pickle.load(f)
with open(ZONES_DIR / "gmm_target.pkl", "rb") as f:
    gmm_targ = pickle.load(f)

# Tokenize all build-ups
token_rows = []

for build_up_id in tqdm(index['build_up_id'].tolist()[:50], desc="Tokenizing"):
    try:
        # Load and preprocess (same as before)
        df = loader.load_build_up(build_up_id)
        meta = loader.get_metadata(build_up_id)
        
        df = prepare_frame_data(df)
        df = enrich_with_team_id(df, meta['game_id'])
        df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))
        df_norm = infer_ball_carrier(df_norm, meta.get('opponent_team_id'))
        df_norm['time_seconds'] = df_norm['time'].apply(time_to_seconds)
        
        pressers = identify_pressers(df_norm, config)
        if len(pressers) == 0:
            continue
        
        # Extract positions
        kick_time = time_to_seconds(str(meta.get('kick_time')))
        t_init = kick_time + 1.0
        t_target = kick_time + 5.0
        
        times = df_norm[['frame', 'time_seconds']].drop_duplicates()
        
        # Initial
        init_frame = times.loc[(times['time_seconds'] - t_init).abs().idxmin(), 'frame']
        init_df = df_norm[df_norm['frame'] == init_frame]
        init_pos = []
        for pid in pressers:
            p = init_df[init_df['player_id'] == pid]
            if not p.empty:
                init_pos.append([p.iloc[0]['x_norm'], p.iloc[0]['y_norm']])
        
        # Target
        target_frame = times.loc[(times['time_seconds'] - t_target).abs().idxmin(), 'frame']
        target_df = df_norm[df_norm['frame'] == target_frame]
        target_pos = []
        for pid in pressers:
            p = target_df[target_df['player_id'] == pid]
            if not p.empty:
                target_pos.append([p.iloc[0]['x_norm'], p.iloc[0]['y_norm']])
        
        if len(init_pos) == 0 or len(target_pos) == 0:
            continue
        
        # Tokenize
        tokens = tokenize_build_up(np.array(init_pos), np.array(target_pos), gmm_init, gmm_targ)
        tokens['build_up_id'] = build_up_id
        token_rows.append(tokens)
    
    except Exception as e:
        continue

# Create term matrix
term_matrix = pd.DataFrame(token_rows)
term_matrix.set_index('build_up_id', inplace=True)
term_matrix.fillna(0.0, inplace=True)

print(f"\nTerm Matrix Shape: {term_matrix.shape}")
print(f"  {term_matrix.shape[0]} build-ups")
print(f"  {term_matrix.shape[1]} tokens (8 x 15 = 120)")
print(f"\nNon-zero tokens per build-up: {(term_matrix > 0).sum(axis=1).mean():.1f}")

# Save
term_matrix.to_parquet(TOKENS_DIR / "term_matrix.parquet")
print(f"Saved to {TOKENS_DIR / 'term_matrix.parquet'}")

### Step 7: Visualize Token Distribution

Which zone transitions are most common?

In [ ]:
# Aggregate token weights across all build-ups
token_sums = term_matrix.sum(axis=0).sort_values(ascending=False)

print("Top 10 Most Frequent Tokens:")
print("Token ID | Total Weight | Init Zone → Target Zone")
print("-" * 55)
for token_name, weight in token_sums.head(10).items():
    token_id = int(token_name.split('_')[1])
    init_zone = token_id // 15
    target_zone = token_id % 15
    print(f"{token_id:4d}     | {weight:12.2f} | Zone {init_zone} → Zone {target_zone}")

# Plot distribution
plt.figure(figsize=(14, 5))
plt.bar(range(len(token_sums.head(20))), token_sums.head(20).values, color='steelblue', edgecolor='black')
plt.xlabel('Token Rank')
plt.ylabel('Total Weight')
plt.title('Top 20 Token Weights Across All Build-Ups')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Part 3: NMF Topic Modeling

### Step 8: Train NMF Model

Discover K pressing pattern topics (default: 10).

In [ ]:
nmf_config = NMFConfig(n_topics=10)

# Train NMF
nmf_model = NMF(
    n_components=nmf_config.n_topics,
    init='nndsvda',
    max_iter=nmf_config.max_iter,
    alpha_W=nmf_config.alpha_W,
    alpha_H=nmf_config.alpha_H,
    l1_ratio=nmf_config.l1_ratio,
    random_state=nmf_config.random_state
)

H = nmf_model.fit_transform(term_matrix.values)  # (build_ups x topics)
W = nmf_model.components_  # (topics x tokens)

print(f"NMF Model Trained:")
print(f"  W matrix: {W.shape} (topics × tokens)")
print(f"  H matrix: {H.shape} (build_ups × topics)")
print(f"  Reconstruction error: {nmf_model.reconstruction_err_:.2f}")
print(f"  Iterations: {nmf_model.n_iter_}")

# Save
with open(TOPICS_DIR / "nmf_model.pkl", "wb") as f:
    pickle.dump(nmf_model, f)

H_df = pd.DataFrame(H, index=term_matrix.index, columns=[f'topic_{i}' for i in range(nmf_config.n_topics)])
H_df.to_parquet(TOPICS_DIR / "build_up_topic_weights.parquet")

print(f"\nSaved to {TOPICS_DIR}/")

### Step 9: Interpret Topics

Examine top tokens per topic to understand pressing patterns.

In [ ]:
def interpret_topic(topic_idx, W, n_top=5):
    """Print top tokens for a topic."""
    topic_weights = W[topic_idx, :]
    top_token_indices = np.argsort(topic_weights)[::-1][:n_top]
    
    print(f"\n{'='*70}")
    print(f"Topic {topic_idx}: Top {n_top} Zone Transitions")
    print(f"{'='*70}")
    print("Token ID | Weight | Init Zone → Target Zone")
    print("-" * 70)
    
    for token_id in top_token_indices:
        init_zone = token_id // 15
        target_zone = token_id % 15
        weight = topic_weights[token_id]
        print(f"{token_id:4d}     | {weight:6.3f} | Zone {init_zone} → Zone {target_zone}")

# Interpret all topics
for topic_idx in range(nmf_config.n_topics):
    interpret_topic(topic_idx, W, n_top=5)

### Step 10: Visualize Topic-Build-Up Associations

Which topics are most prevalent? How are topics distributed across build-ups?

In [ ]:
# Topic prevalence
topic_prevalence = H_df.sum(axis=0).sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Topic prevalence bar chart
ax1.bar(range(len(topic_prevalence)), topic_prevalence.values, color='teal', edgecolor='black')
ax1.set_xlabel('Topic ID')
ax1.set_ylabel('Total Weight Across Build-Ups')
ax1.set_title('Topic Prevalence')
ax1.set_xticks(range(len(topic_prevalence)))
ax1.set_xticklabels([f'T{i}' for i in range(nmf_config.n_topics)])
ax1.grid(alpha=0.3, axis='y')

# Topic distribution heatmap (first 20 build-ups)
sns.heatmap(H_df.head(20).T, cmap='YlOrRd', cbar_kws={'label': 'Weight'}, 
            linewidths=0.5, ax=ax2)
ax2.set_xlabel('Build-Up ID')
ax2.set_ylabel('Topic')
ax2.set_title('Topic Weights per Build-Up (First 20)')

plt.tight_layout()
plt.show()

print(f"\nMost prevalent topic: Topic {topic_prevalence.idxmax().split('_')[1]}")
print(f"Least prevalent topic: Topic {topic_prevalence.idxmin().split('_')[1]}")

### Step 11: Dominant Topic per Build-Up

Assign each build-up to its strongest topic.

In [ ]:
# Find dominant topic per build-up
dominant_topics = H_df.idxmax(axis=1)
dominant_topic_counts = dominant_topics.value_counts().sort_index()

print("Build-Ups per Dominant Topic:")
print(dominant_topic_counts)

plt.figure(figsize=(10, 5))
dominant_topic_counts.plot(kind='bar', color='coral', edgecolor='black')
plt.xlabel('Dominant Topic')
plt.ylabel('Number of Build-Ups')
plt.title('Build-Up Clustering by Dominant Topic')
plt.xticks(rotation=0)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Summary

You've learned how to:
1. ✅ Collect presser positions from build-ups
2. ✅ Train GMM zone models (8 initial, 15 target zones)
3. ✅ Visualize learned zones as ellipses on the pitch
4. ✅ Tokenize zone transitions (120 discrete tokens)
5. ✅ Build term matrix (build-ups × tokens)
6. ✅ Train NMF model to discover pressing pattern topics
7. ✅ Interpret topics by examining top tokens
8. ✅ Assign build-ups to dominant topics

## Next Steps

- **Tutorial 4**: Create visualizations (heatmaps, animations, network graphs)
- **Analysis**: Compare feature distributions across topics
- **Tactical Insights**: Generate narrative descriptions of topics

## Key Takeaways

- **GMM zones** provide interpretable spatial discretization
- **Tokenization** converts movements to discrete sequences
- **NMF topics** reveal recurring pressing patterns (e.g., left-side press, high line, containment)
- Topics can be linked to match context (opponent, score, time) for deeper analysis